# Phase 1c — IDRiD Part B grading images (top-up)

Caches the **516 Part B images** (`IDRiD_001`–`516`) beside the Part A masks already in
`verify-dr-idrid-masks`. Small and self-contained: ~600 images, a few minutes, no GPU.

## Why C1 could not run

IDRiD publishes two different image sets, and their names look deceptively alike:

| Part | Images | Named | Carries |
|---|---|---|---|
| A — Segmentation | 81 | `IDRiD_01`–`81` | lesion masks → **C2** |
| B — Disease Grading | 516 | `IDRiD_001`–`516` | grades, and the Part C centres → **C1** |

`IDRiD_01` is not `IDRiD_001`. The cache held Part A only, so the Part C optic-disc and
fovea coordinates had no image to attach to, and **C1 had zero training targets**.

Two things let that pass unnoticed, and both are now fixed:

1. `02_manifests.ipynb` picked `--coords-source-dir` by looking for `"segmentation"` in
   the path, which pinned it to Part A. Every coordinate row missed. It now resolves the
   directory by **matching the coordinate tables' own IDs**, and prints the match counts.
2. When the grades failed to join, Phase 2 fell back to `--no-grades` and the manifest
   built anyway — a successful-looking run with an empty geometry column.

## Inputs

The raw IDRiD mount, plus `verify-dr-idrid-masks` so Part A's masks are carried
forward into the republished cache.

## Settings

| Setting | Value |
|---|---|
| Accelerator | **None** |
| Persistence | Files only |
| Internet | **On** |

## 1 · Pull the code

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

In [ ]:
from pathlib import Path
import os, re, json

INPUT = Path("/kaggle/input")

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace("%20", "")

def dataset_roots(max_depth=3):
    """Candidate dataset directories, shallowest first.

    Kaggle does not always mount datasets as direct children of /kaggle/input --
    they can sit under competitions/ and datasets/ wrappers. Breadth-first so a
    shallower match always wins over a nested subfolder of the same name.
    """
    level, out = [INPUT], []
    for _ in range(max_depth):
        nxt = []
        for d in level:
            try:
                children = sorted(c for c in d.iterdir() if c.is_dir())
            except OSError:
                continue
            out.extend(children)
            nxt.extend(children)
        level = nxt
    return out

EXCLUDE_ROOTS = []      # set to [CACHE] once the cache is located

def _under(path, root):
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False

def find_mount(*keywords, required=True):
    """Locate a mounted dataset by keyword, so a renamed mirror doesn't break the notebook.

    Anything under EXCLUDE_ROOTS is skipped. The cache contains directories named
    ddr, eyepacs, idrid, aptos and messidor2 -- exactly the keywords searched for --
    so without this a raw-dataset lookup can land inside the cache instead.
    """
    for d in dataset_roots():
        if any(_under(d, r) for r in EXCLUDE_ROOTS):
            continue
        if all(_norm(k) in _norm(d.name) for k in keywords):
            return d
    if required:
        print(f"  !! NOT MOUNTED: {' + '.join(keywords)}")
        print("     Use 'Add Input' in the right-hand panel. Candidate dirs seen:")
        for d in dataset_roots(2)[:25]:
            print(f"       - {d.relative_to(INPUT)}")
    return None

def find_any(*keyword_sets, label="", required=True):
    """Try several keyword spellings. Mirrors title themselves inconsistently:
    IDRiD ships as 'idrid-dataset' or 'indian-diabetic-retinopathy-image-dataset'."""
    for kws in keyword_sets:
        hit = find_mount(*kws, required=False)
        if hit:
            return hit
    if required:
        print(f"  !! NOT MOUNTED: {label or keyword_sets[0]}")
        print("     Currently mounted:")
        for d in sorted(INPUT.iterdir()):
            print(f"       - {d.name}")
    return None

def match_channel(dirname):
    """Map a mask directory name to a lesion channel.

    DDR uses MA/HE/EX/SE; IDRiD uses '1. Microaneurysms', '2. Haemorrhages',
    '3. Hard Exudates', '4. Soft Exudates', '5. Optic Disc'. Match on meaning so
    one function covers both.
    """
    n = re.sub(r"^\d+\.\s*", "", dirname.lower().strip())
    n = n.replace("%20", " ")
    if n == "ma" or "microaneurysm" in n:
        return "microaneurysm"
    if n == "he" or "haemorrhage" in n or "hemorrhage" in n:
        return "haemorrhage"
    if n == "ex" or ("hard" in n and "exudate" in n):
        return "hard_exudate"
    if n == "se" or ("soft" in n and "exudate" in n) or "cotton" in n:
        return "soft_exudate"
    if n == "od" or "optic disc" in n:
        return "optic_disc"
    return None

MASK_EXT = {".tif", ".tiff", ".png", ".gif", ".bmp", ".jpg", ".jpeg"}
LESION4 = ["microaneurysm", "haemorrhage", "hard_exudate", "soft_exudate"]

def scan_mask_dirs(root, require=""):
    """Find mask directories under root, keyed by lesion channel."""
    found = {}
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if require and require not in str(d).lower():
            continue
        channel = match_channel(d.name)
        if not channel:
            continue
        files = [f for f in d.rglob("*") if f.suffix.lower() in MASK_EXT]
        if files:
            found.setdefault(channel, []).append((str(d), len(files)))
    return found

def du(path, cap=40000):
    """Rough size + file count, capped so it stays fast on huge mounts."""
    total = n = 0
    for i, f in enumerate(Path(path).rglob("*")):
        if i > cap:
            return total, n, True
        if f.is_file():
            total += f.stat().st_size
            n += 1
    return total, n, False

print("helpers ready")

import shlex, subprocess

CACHE = Path("/kaggle/working/cache512")

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    print("$", cmd[:150] + (" ..." if len(cmd) > 150 else ""))
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(proc.stdout.rstrip())
    if proc.returncode != 0:
        print(proc.stderr.rstrip())
        raise RuntimeError(f"command failed with exit {proc.returncode}")
    return proc.stdout

## 2 · Carry the existing IDRiD cache forward

A Kaggle dataset version is a **complete snapshot**, so publishing from this notebook
without Part A's masks would delete them. They are copied into `/kaggle/working` first
and republished together.

In [ ]:
import shutil
from pathlib import Path

# Any mounted dataset holding <something>/idrid/masks with files in it. Scanned
# directly rather than through the Phase 2 resolver, which is not defined here.
prior = None
for cand in sorted(INPUT.glob("*/**/idrid/masks")):
    if cand.is_dir() and any(cand.iterdir()):
        prior = cand.parent
        break
if prior is None:
    for cand in sorted(INPUT.glob("*/**/IDRiD/masks")):
        if cand.is_dir() and any(cand.iterdir()):
            prior = cand.parent
            break

# Keep this project's OWN published datasets out of raw-dataset discovery.
# verify-dr-idrid-masks normalises to "verifydridridmasks", which contains
# "idrid" -- so without this, find_any(("idrid",)) can resolve to our cache
# instead of the raw IDRiD mount, and which one wins is down to sort order.
for cand in sorted(INPUT.iterdir()):
    if cand.is_dir() and _norm(cand.name).startswith("verifydr"):
        if cand not in EXCLUDE_ROOTS:
            EXCLUDE_ROOTS.append(cand)
if prior is not None and prior.parent not in EXCLUDE_ROOTS:
    EXCLUDE_ROOTS.append(prior.parent)
print("excluded from raw discovery:", *[str(r) for r in EXCLUDE_ROOTS] or ["(none)"], sep="\n   ")
print()

if prior is None:
    print("!! No existing IDRiD cache with masks found.")
    print("!! Attach verify-dr-idrid-masks, or run 01b_idrid_masks.ipynb first.")
    print("!! Continuing anyway: this notebook will cache Part B alone, and C2")
    print("!! will then need the masks dataset attached separately.")
else:
    dest = CACHE / "idrid"
    dest.mkdir(parents=True, exist_ok=True)
    print(f"copying the existing IDRiD cache from {prior}")
    for item in prior.iterdir():
        target = dest / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
    n_img = sum(1 for _ in (dest / "images").rglob("*")) if (dest / "images").is_dir() else 0
    chans = sorted(d.name for d in (dest / "masks").iterdir()) if (dest / "masks").is_dir() else []
    print(f"  carried forward: {n_img} images, mask channels {chans}")

## 3 · Find Part B

Located by the coordinate tables' own IDs rather than by directory name, so a mirror
that renames its folders still resolves correctly.

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, str(REPO_DIR / "src"))
from verify_dr.data.idrid_tables import find_coord_tables, report

IMG_EXT = {".jpg", ".jpeg", ".png", ".tif"}

# Pick the mount by CONTENT, not by the first name match. More than one IDRiD
# mirror is commonly attached -- "idrid-..." and "indian-diabetic-retinopathy-..."
# are the same dataset under two names, and some mirrors ship Part A only. The
# one that matters is whichever actually carries Part C tables.
candidates = []
for d in sorted(INPUT.iterdir()):
    if not d.is_dir() or any(_under(d, r) for r in EXCLUDE_ROOTS):
        continue
    name = _norm(d.name)
    if "idrid" in name or all(k in name for k in ("diabetic", "retinopathy")):
        candidates.append(d)

assert candidates, (
    "No raw IDRiD mount found. Add the original IDRiD dataset with 'Add Input' -- "
    "verify-dr-idrid-masks is this project's own cache and holds no tables.\n"
    "Attached inputs: " + ", ".join(sorted(x.name for x in INPUT.iterdir())))

print(f"{len(candidates)} candidate IDRiD mount(s). Scoring by what each contains:")
print()
scored_mounts = []
for d in candidates:
    found, not_coords, unreadable = find_coord_tables(d)
    ids = set().union(*[i for i, _c, _xy in found.values()]) if found else set()
    print(f"--- {d.name}  ({len(ids)} coordinate ids)")
    for line in report(found, not_coords, unreadable).splitlines():
        print("   " + line)
    print()
    scored_mounts.append((len(ids), d, found, ids))

scored_mounts.sort(key=lambda t: -t[0])
_n, idrid, found, coord_ids = scored_mounts[0]
coord_tables = sorted(found)

if not coord_ids:
    print("=" * 70)
    print("No coordinate table resolved in ANY candidate mount.")
    for _n, d, _f, _i in scored_mounts:
        tables = sorted(f for f in d.rglob("*")
                        if f.is_file() and f.suffix.lower() in {".csv", ".xlsx", ".xls"})
        print(f"\n  {d.name}: {len(tables)} table file(s)")
        for f in tables[:15]:
            print("     ", f.relative_to(d))
        if not tables:
            print("      top-level contents:")
            for x in sorted(d.iterdir())[:15]:
                print("        ", x.name + ("/" if x.is_dir() else ""))
    print("=" * 70)

assert coord_ids, (
    "No Part C coordinate table could be read from any attached IDRiD mount. The "
    "listing above shows, per mount, which case applies: no tables present (wrong "
    "dataset), present but unreadable (reason printed, with an openpyxl hint when "
    "that is it), or readable but not coordinate tables.")

print("=" * 70)
print(f"using {idrid.name}  --  {len(coord_ids)} images have centres")
print("=" * 70)

image_dirs = sorted(
    d for d in idrid.rglob("*")
    if d.is_dir() and any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file()))

scored = []
for d in image_dirs:
    stems = {f.stem for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT}
    hits = len(stems & coord_ids)
    if hits:
        scored.append((hits, len(stems), d))
scored.sort(reverse=True)

print()
print(f"Part C names {len(coord_ids)} images. Directories holding them:")
for hits, total, d in scored:
    print(f"   {hits:>4} of {total:<4} {d}")

if not scored:
    print("   NONE. Image directories searched:")
    for d in image_dirs[:20]:
        stems = sorted(f.stem for f in d.iterdir()
                       if f.is_file() and f.suffix.lower() in IMG_EXT)
        print(f"      {len(stems):>5} images, e.g. {stems[:2]}  {d}")
    print(f"   Coordinate ids look like: {sorted(coord_ids)[:3]}")

assert scored, (
    "No directory contains the images the coordinate tables name. Compare the stems "
    "above against the coordinate ids: Part A is IDRiD_01-81 and Part B is "
    "IDRiD_001-516, and only Part B is named by the Part C tables.")

part_b = [d for _h, _t, d in scored]

## 4 · Cache Part B

Same `--size 512 --fit pad` as Phase 1, into the same `idrid` dataset directory.
`build_cache.py` resumes, so Part A's images are skipped rather than re-encoded.

In [ ]:
for d in part_b:
    print(f"\n=== IDRiD Part B: {d.name} ===")
    run(f"python {q(REPO_DIR / 'scripts/build_cache.py')} "
        f"--source-root {q(d)} --dataset IDRiD --output-root {q(CACHE)} "
        f"--size 512 --fit pad --contact-sheet 40")

## 5 · The gate

Checks the one thing that actually decides whether C1 can run: **how many cached images
are named by the coordinate tables.** The previous cache passed every check it had and
still scored 0 here.

In [ ]:
import json

cache_idrid = CACHE / "idrid"
cached = {f.stem for f in (cache_idrid / "images").rglob("*")
          if f.is_file() and f.suffix.lower() in IMG_EXT}
joinable = cached & coord_ids

mask_root = cache_idrid / "masks"
channels = {d.name: len(list(d.glob("*"))) for d in mask_root.iterdir()} if mask_root.is_dir() else {}

print(f"cached images            {len(cached)}")
print(f"named by Part C          {len(joinable)}   <-- C1's training set")
print(f"mask files per channel   {channels}")
print()

ok = True
if len(joinable) < 100:
    ok = False
    print("FAIL - too few images carry coordinates. C1 cannot train.")
    print("       Check section 3: did Part B resolve to a real directory?")
else:
    print(f"PASS - {len(joinable)} images carry optic-disc and fovea centres.")

lesion = {c: n for c, n in channels.items() if c != "optic_disc"}
if not lesion or all(n == 0 for n in lesion.values()):
    ok = False
    print("FAIL - no lesion masks in this cache. Section 2 did not carry them forward,")
    print("       so publishing this would delete them. Attach verify-dr-idrid-masks.")
else:
    print(f"PASS - lesion masks intact: {lesion}")

assert ok, "do not publish this cache - fix the failures above first"

## 6 · Contact sheet

In [ ]:
from IPython.display import Image, display
for sheet in sorted(CACHE.rglob("contact_sheet.jpg")):
    print(sheet)
    display(Image(str(sheet)))

---
## 7 · Publish

**Save Version → Save & Run All (Commit).** Then Output tab → **New Dataset**, named
**`verify-dr-idrid`**.

> A new dataset, not a new version of `verify-dr-idrid-masks` — that one is linked to
> `01b_idrid_masks.ipynb`, and a version made from a different notebook replaces the
> whole snapshot.

This cache **supersedes `verify-dr-idrid-masks`**: it holds Part A's masks *and* Part B's
images. Attach `verify-dr-idrid` in place of it from here on.

Then:

1. Re-run **`02_manifests.ipynb`** with `verify-dr-idrid` attached. Section 6 now prints
   how many IDs each candidate directory matched — expect a directory matching ~516.
   Confirm `coordinates projected for N images` with N in the hundreds, **not 0**.
2. Re-run **`04_phase4.ipynb`**. Section 5 should report C1 **READY**.